# 13 • Agent Loop: ประสานงานคลังและขนส่ง

เป้าหมาย: ดู Decide → Tool → Observe หลายรอบ เพื่อเชื่อมข้อมูลระดับน้ำกับ กฏ และหยุดเมื่อได้คำตอบหรือครบจำนวนรอบ

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ กฏ ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

[![Open in Colab](https://img.shields.io/badge/Open_in_Colab-F9AB00?style=flat-square&logo=googlecolab&logoColor=white)](https://colab.research.google.com/drive/1kAVsZz4z_OOrxrcqCE6CKzYhQQ0penyM?usp=drive_link)


## เตรียมพร้อมก่อนเริ่ม
แทนที่ `your_api_key` ในตัวแปร `OPENAI_API_KEY` ด้วย API key จริงของคุณ แล้วรัน cells จากบนลงล่าง ใช้รูปแบบเดียวกันทั้ง local และ Colab โดย cell ติดตั้ง packages อยู่ใน notebook แล้ว

ก่อนแชร์หรือ commit notebook ให้เปลี่ยน API key กลับเป็น `your_api_key`


In [ ]:
%pip install -q openai numpy

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"


In [ ]:
chunks = [
    "[กฏ-1 รับน้ำยาง] บันทึก Lot ผู้ขาย น้ำหนักสุทธิ ทะเบียนรถ และเวลารับเข้า เก็บรหัสผู้ขายเพื่อเชื่อมโยงข้อมูล Traceability",
    "[กฏ-2 QC] เก็บผล DRC พร้อมรหัสตัวอย่างและ Lot หากผลยังไม่ครบ ให้ระบุสถานะรอ QC ห้ามสรุปว่าผ่านเกณฑ์จากข้อมูลที่หายไป",
    "[กฏ-3 คลังสินค้า] Lot ที่รอ QC ต้องแยกในพื้นที่พักรอผลและติดป้าย Lot ตรวจสถานะก่อนยืนยันการจัดส่ง",
    "[กฏ-4 ประสานงาน] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 30 ซม. ให้ประสานหัวหน้าคลังและฝ่ายขนส่งเพื่อตรวจสอบแผนรับส่งสินค้า",
    "[กฏ-5 น้ำท่วม] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 50 ซม. ให้พักแผนโหลดสินค้าของพื้นที่ที่ได้รับผลกระทบ แจ้งหัวหน้าคลัง ฝ่ายขนส่ง และ EHS เพื่อประเมินตามแผนของพื้นที่",
    "[กฏ-6 ส่งมอบ] เมื่อเวลารับส่งเปลี่ยน ให้ฝ่ายขนส่งประสานฝ่ายขาย แจ้งกำหนดการใหม่ที่ยืนยันแล้ว ห้ามรับรองเวลาส่งมอบที่ยังไม่ยืนยัน",
    "[กฏ-7 ซ่อมบำรุง] เมื่อพบปัญหาเครื่องจักร ให้บันทึกอาการ รหัสเครื่อง และเวลา แล้วส่งใบงานให้ผู้รับผิดชอบซ่อมบำรุง",
    "[กฏ-8 สิ่งแวดล้อม] หากพบข้อร้องเรียนด้านกลิ่นหรือน้ำเสีย ให้บันทึกเวลา จุดที่พบ และรายละเอียด ส่งต่อผู้รับผิดชอบสิ่งแวดล้อมเพื่อตรวจสอบ"
]

In [ ]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(texts):
    result = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([item.embedding for item in result.data])

vectors = embed(chunks)
print("ขนาด Embedding matrix:", vectors.shape)

In [ ]:
def retrieve(question, k=2):
    query = embed([question])[0]
    scores = vectors @ query / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query))
    indices = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in indices]

In [ ]:
def get_flood_level(factory: str):
    """อ่านระดับน้ำสมมติของคลังยางแท่ง A หรือคลังน้ำยาง B สำหรับการอบรม"""
    readings = {"คลังยางแท่ง A": 55, "คลังน้ำยาง B": 10}
    if factory not in readings:
        return {"error": "ไม่พบหน่วยงานนี้", "source": "ข้อมูลสมมติสำหรับการอบรม"}
    return {"factory": factory, "water_cm": readings[factory], "source": "ข้อมูลสมมติสำหรับการอบรม", "live": False}

In [ ]:
def search_rules(question):
    # ดึง 3 ข้อเพื่อเปรียบเทียบแนวทางประสานงานและเกณฑ์ระดับน้ำ
    return [text for text, score in retrieve(question, k=3)]

functions = {"get_flood_level": get_flood_level, "search_rules": search_rules}

In [ ]:
tools = [{
    "type": "function", "name": "get_flood_level",
    "description": "อ่านระดับน้ำสมมติของคลังยางแท่ง A หรือคลังน้ำยาง B",
    "parameters": {"type": "object", "properties": {
        "factory": {"type": "string", "enum": ["คลังยางแท่ง A", "คลังน้ำยาง B"]}
    }, "required": ["factory"], "additionalProperties": False},
    "strict": True,
}]

tools.append({
    "type": "function", "name": "search_rules",
    "description": "ค้นข้อความ กฏ สมมติที่เกี่ยวข้องกับคำถามงานคลังยาง",
    "parameters": {"type": "object", "properties": {"question": {"type": "string"}},
                   "required": ["question"], "additionalProperties": False},
    "strict": True,
})

In [ ]:
import json

def run_agent(question, instructions):
    messages = [{"role": "user", "content": question}]
    observed = None
    searched = False
    for step in range(5):
        # บังคับลำดับ observation: วัดสถานะ -> ค้น กฏ -> ตอบ
        name = "get_flood_level" if observed is None else "search_rules" if not searched else None
        available = [tool for tool in tools if tool["name"] == name] if name else tools
        response = client.responses.create(
            model=MODEL, input=messages, instructions=instructions,
            tools=available, tool_choice="required" if name else "auto",
            parallel_tool_calls=False, max_output_tokens=500,
        )
        messages += response.output
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            print(response.output_text)
            return response.output_text
        for call in calls:
            args = json.loads(call.arguments)
            if call.name == "search_rules" and observed:
                # ใช้ค่าที่ได้จริงจาก tool ไม่พึ่งให้โมเดลจำตัวเลขเอง
                args["question"] = f"น้ำท่วม ระดับน้ำ {observed['water_cm']} ซม. " + args["question"]
            print(f"รอบที่ {step + 1}:", call.name, args)
            try:
                result = functions[call.name](**args)
            except Exception as error:
                print("Tool ล้มเหลว:", str(error))
                return
            if call.name == "get_flood_level":
                if "water_cm" not in result:
                    print("ไม่พบข้อมูลระดับน้ำ:", result)
                    return
                observed = result
            elif call.name == "search_rules":
                searched = True
            print("ผลที่สังเกตได้:", result)
            messages.append({"type": "function_call_output", "call_id": call.call_id,
                             "output": json.dumps(result, ensure_ascii=False)})
    print("หยุดเมื่อครบ 5 รอบ ยังไม่ได้คำตอบสุดท้าย")

In [ ]:
instructions = 'คุณเป็น Agent ผู้ช่วยประสานงานคลังยางสำหรับการอบรม ตอบเป็นภาษาไทย\nตรวจระดับน้ำสมมติก่อน แล้วค้น กฏ โดยระบุค่าระดับน้ำและคำถามงานคลัง/ขนส่ง\nแยกข้อมูลจาก tool ออกจากแนวทางในเอกสาร ระบุว่าค่าเป็นข้อมูลสมมติ อ้างรหัส กฏ\nห้ามสร้างผล tool หรือกำหนดส่งมอบเอง หาก tool ล้มเหลวให้บอกข้อจำกัด'

answer = run_agent("ตรวจระดับน้ำสมมติของคลังยางแท่ง A แล้วค้น กฏ เพื่อประสานงานคลังและขนส่ง", instructions)

## ลองทำด้วยตนเอง
1. เปลี่ยนเป็นคลังน้ำยาง B แล้วตรวจค่าที่อ่านและ กฏ ที่ Agent เลือก
2. อธิบายความต่างระหว่าง instructions กับการบังคับลำดับด้วย tool_choice และ parallel_tool_calls=False
3. จำกัด loop เหลือ 1 รอบ แล้วดูว่าทำไมยังไม่มีคำตอบสุดท้าย

ตัวอย่างนี้เป็น Agent Loop ที่มีลำดับงานจำกัด: ต้องอ่านระดับน้ำก่อนค้น กฏ การค้นใช้ค่าที่ tool อ่านได้จริง แต่ยังต้องตรวจว่าข้อกำหนดใน กฏ ใช้กับสถานการณ์นั้นหรือไม่
